# Medicare Benefits — LLM-First Notebook (No benefit_rules)
### LangChain + Azure OpenAI

**Input:** `medicare_input_loadid_142.json` — PBP rows only (no benefit_rules needed)  
**Prompts:** loaded from the three `.txt` files alongside this notebook  
**Output:** 10-column benefit description CSV

| Cell | Does |
|---|---|
| 1 | Load input JSON (PBP rows only) |
| 2 | Configure Azure OpenAI |
| 3 | Load prompts from `.txt` files |
| 4 | Build chain + helpers |
| 5 | Run pipeline (one LLM call) |
| 6 | Parse, export CSV, inspect results |
| 7 | Reload prompts after editing |


---
## Cell 1 — Load input JSON (PBP rows only)

In [ ]:
import json, os, re

INPUT_JSON_PATH = "medicare_input_loadid_142.json"

with open(INPUT_JSON_PATH, encoding="utf-8") as f:
    input_json = json.load(f)

# Support both formats:
#   Format A: {"pbp": [...], "benefit_rules": [...]}  (old format — pbp key used, rules ignored)
#   Format B: just the raw list of PBP rows            (new format)
if isinstance(input_json, dict) and "pbp" in input_json:
    pbp_rows = input_json["pbp"]
elif isinstance(input_json, list):
    pbp_rows = input_json
else:
    raise ValueError("Unexpected JSON format — expected list of PBP rows or dict with 'pbp' key")

load_id = str(pbp_rows[0].get("LoadID", "unknown")) if pbp_rows else "unknown"

print(f"Loaded: {INPUT_JSON_PATH}")
print(f"  load_id:  {load_id}")
print(f"  pbp rows: {len(pbp_rows):,}")
print()
print("Sample row:")
print(json.dumps(pbp_rows[0], indent=2))


---
## Cell 2 — Configure Azure OpenAI

In [ ]:
# !pip install langchain langchain-openai langchain-core openai pandas   # run once

import pandas as pd

try:
    from langchain_openai import AzureChatOpenAI
except Exception:
    from langchain.chat_models import AzureChatOpenAI

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT",    "https://YOUR-RESOURCE.cognitiveservices.azure.com/")
AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY",     "YOUR_API_KEY")
AZURE_OPENAI_DEPLOYMENT  = os.getenv("AZURE_OPENAI_DEPLOYMENT",  "gpt-4o")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
CARRIER_PREFIX = "MOM"

print(f"Model:    {AZURE_OPENAI_DEPLOYMENT}")
print(f"Endpoint: {AZURE_OPENAI_ENDPOINT}")


---
## Cell 3 — Load prompts from `.txt` files

Edit the three files freely — re-run this cell (no kernel restart needed).

| File | Contains |
|---|---|
| `system_prompt.txt` | Benefit ID reference, formatting rules, output schema |
| `few_shot_examples.txt` | 15 worked PBP→output examples (replaces benefit_rules) |
| `human_template.txt` | Request structure with `{placeholders}` |


In [ ]:
PROMPT_DIR = "."  # folder containing the .txt files

def load_prompt(filename):
    with open(os.path.join(PROMPT_DIR, filename), encoding="utf-8") as f:
        return f.read()

SYSTEM_PROMPT  = load_prompt("system_prompt.txt")
FEW_SHOT       = load_prompt("few_shot_examples.txt")
HUMAN_TEMPLATE = load_prompt("human_template.txt")

# Merge few_shot into system prompt so LangChain never scans
# {{token}} patterns in examples as missing template variables.
SYSTEM_PROMPT_WITH_EXAMPLES = SYSTEM_PROMPT + "\n\n" + FEW_SHOT
HUMAN_TEMPLATE = HUMAN_TEMPLATE.replace("{few_shot}\n\n", "").replace("{few_shot}", "")

# Sanity check placeholders
REQUIRED = ["{carrier_prefix}", "{plan_type}", "{file_name}", "{n_pbp}", "{pbp_json}"]
missing  = [p for p in REQUIRED if p not in HUMAN_TEMPLATE]
if missing:
    print(f"WARNING: human_template.txt missing placeholders: {missing}")
else:
    print("All required placeholders found")

print(f"  system_prompt.txt      {len(SYSTEM_PROMPT):>7,} chars")
print(f"  few_shot_examples.txt  {len(FEW_SHOT):>7,} chars")
print(f"  human_template.txt     {len(HUMAN_TEMPLATE):>7,} chars")
print(f"  combined system msg    {len(SYSTEM_PROMPT_WITH_EXAMPLES):>7,} chars")


---
## Cell 4 — Build LangChain chain + helpers

In [ ]:
llm = AzureChatOpenAI(
    azure_endpoint   = AZURE_OPENAI_ENDPOINT,
    api_key          = AZURE_OPENAI_API_KEY,
    azure_deployment = AZURE_OPENAI_DEPLOYMENT,
    api_version      = AZURE_OPENAI_API_VERSION,
    temperature      = 0,
    max_tokens       = 16000,
)


def invoke_chain(template_vars: dict) -> str:
    """
    Fill HUMAN_TEMPLATE with template_vars via str.format_map(),
    then call LLM directly — no ChatPromptTemplate so no brace-scanning KeyErrors.
    """
    human_text = HUMAN_TEMPLATE.format_map(template_vars)
    response   = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT_WITH_EXAMPLES),
        HumanMessage(content=human_text),
    ])
    return response.content


def extract_plan_meta(pbp_rows: list) -> dict:
    """Extract planid, plan_type, file_name from PBP rows."""
    file_name = pbp_rows[0].get("FileName", "").strip() if pbp_rows else ""
    remainder = file_name[file_name.index("_") + 1:] if "_" in file_name else file_name
    parts     = remainder.split("-")
    contract  = parts[0] if len(parts) > 0 else ""
    plan      = parts[1] if len(parts) > 1 else ""
    seg       = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else 0
    planid    = f"{CARRIER_PREFIX}_{contract}_{plan}_{seg}"
    plan_type = "HMO"
    for r in pbp_rows:
        if r.get("header") == "Plan Characteristics" and r.get("field") == "Plan Type":
            plan_type = r.get("value", "HMO").strip()
            break
    return {"file_name": file_name, "planid": planid, "plan_type": plan_type}


def parse_llm_output(raw: str) -> list:
    """Extract and parse the JSON array from the LLM response."""
    text  = re.sub(r"^```[a-z]*\n?", "", raw.strip(), flags=re.MULTILINE)
    text  = re.sub(r"\n?```$",       "", text,         flags=re.MULTILINE).strip()
    start = text.find("[")
    end   = text.rfind("]") + 1
    if start == -1 or end == 0:
        raise ValueError("No JSON array in LLM response")
    return json.loads(text[start:end])


print("Chain and helpers ready")


---
## Cell 5 — Run pipeline
One LangChain call processes all PBP rows and returns all benefit rows.


In [ ]:
meta = extract_plan_meta(pbp_rows)

print(f"File:      {meta['file_name']}")
print(f"Plan ID:   {meta['planid']}")
print(f"Plan type: {meta['plan_type']}")
print(f"PBP rows:  {len(pbp_rows):,}")
print()
print("Calling Azure OpenAI...")

raw_response = invoke_chain({
    "carrier_prefix": CARRIER_PREFIX,
    "file_name":      meta["file_name"],
    "plan_type":      meta["plan_type"],
    "n_pbp":          len(pbp_rows),
    "pbp_json":       json.dumps(pbp_rows, indent=2),
})

print(f"Response received ({len(raw_response):,} chars)")


---
## Cell 6 — Parse, export CSV, inspect

In [ ]:
COLS = [
    "planid", "plantypeid", "benefitid", "benefitname",
    "coveragetypeid", "coveragetypedesc",
    "serviceTypeID", "serviceTypeDesc",
    "benefitdesc", "tinyDescription",
]

output_rows = parse_llm_output(raw_response)

df = (
    pd.DataFrame(output_rows)
    .reindex(columns=COLS)
    .sort_values(["benefitid", "serviceTypeID"])
    .reset_index(drop=True)
)

CSV_OUT = f"output_benefits_loadid_{load_id}.csv"
df.to_csv(CSV_OUT, index=False)

print(f"{len(df)} rows written to {CSV_OUT}")
df[["benefitid", "benefitname", "serviceTypeDesc", "tinyDescription"]]


In [ ]:
# Inspect a specific benefit in full
INSPECT_ID = 1900  # change to any benefitid

for _, r in df[df["benefitid"].astype(str) == str(INSPECT_ID)].iterrows():
    print(f"[{r.benefitid}] {r.benefitname}  svc={r.serviceTypeDesc}")
    print(f"  benefitdesc:     {r.benefitdesc}")
    print(f"  tinyDescription: {r.tinyDescription}")
    print()


---
## Cell 7 — Reload prompts after editing
Edit any `.txt` file then run this cell — no kernel restart needed.


In [ ]:
SYSTEM_PROMPT  = load_prompt("system_prompt.txt")
FEW_SHOT       = load_prompt("few_shot_examples.txt")
HUMAN_TEMPLATE = load_prompt("human_template.txt")

SYSTEM_PROMPT_WITH_EXAMPLES = SYSTEM_PROMPT + "\n\n" + FEW_SHOT
HUMAN_TEMPLATE = HUMAN_TEMPLATE.replace("{few_shot}\n\n", "").replace("{few_shot}", "")

print("Prompts reloaded")
print(f"  system_prompt.txt      {len(SYSTEM_PROMPT):>7,} chars")
print(f"  few_shot_examples.txt  {len(FEW_SHOT):>7,} chars")
print(f"  combined system msg    {len(SYSTEM_PROMPT_WITH_EXAMPLES):>7,} chars")
